# L04 · 机器人模型、DOF 与关节控制

**从模型名称到实测运动**

本实验第一次加载完整 Franka。你会先检查它的运行时结构，再发送控制命令；随后测量一个关节目标怎样经过位置控制器、机器人动力学和执行器限幅变成运动。

## 本实验的产出

- 一份运行时 Link/Joint/DOF/qpos 清单和名称到 local DOF 的映射表；
- 一次 joint4 基准运动，包括初末姿态证据以及同步的 q、qdot 和 control-force 轨迹；
- 三组受控 KP/KV 实验，以及从当前数据动态计算的阶跃响应指标；
- 根据当前数组生成的四段式 Guided interpretation；
- 对结构、时间、重置、控制、饱和和所选视觉路径的明确 PASS/FAIL 证据。

运行每组实验前先写下预测。“没有异常”和一张看似合理的最终图像都不足以作为证据。


## 运行前准备

默认的 `ROBO_GENESIS_BACKEND=auto` 路径会在经过验证的 AMD 后端可用时选择它，否则使用 CPU。需要明确请求最低 CPU 路径时，设置 `ROBO_GENESIS_BACKEND=cpu`。notebook 会同时打印请求和实际使用的后端。

渲染是一项独立能力。保留 `ROBO_GENESIS_RENDER=0` 时，实验运行数值路径，并根据实测 Link 位置绘制姿态示意图。在启动 kernel 前把它设置为 `1`，才会在 `build()` 前声明 Genesis camera，并要求得到有效的初末 RGB 帧。已请求的渲染如果失败，notebook 会停止，不能静默把替代图标成 camera 证据。

setup cell 只调用一次 `gs.init()`。重新运行该 cell，或者修改后端、渲染和任何 build-time 设置前，必须重启 kernel。图表会写入 `ROBO_GENESIS_OUTPUTS_DIR` 指定的目录；未设置时写入仓库的 `outputs/`。本实验不需要网络下载或 YCB 资产。


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.course_utils import environment_report, notebook_mode, select_backend, to_numpy
from robo_genesis.scene_config import (
    FRANKA_FORCE_MAX,
    FRANKA_FORCE_MIN,
    FRANKA_KP,
    FRANKA_KV,
    FRANKA_MJCF,
    FRANKA_QPOS,
)

lesson = load_course_manifest().lesson("L04")
assert lesson.slug == "robot-models-dofs-and-joint-control"
assert lesson.status.value == "cpu-verified"

backend_mode = os.environ.get("ROBO_GENESIS_BACKEND", "auto").strip().lower()
if backend_mode not in {"auto", "cpu"}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")

render_value = os.environ.get("ROBO_GENESIS_RENDER", "0").strip()
if render_value not in {"0", "1"}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == "1"

runtime = notebook_mode("l04-joint-control", show_viewer=False)
output_dir = runtime["output_dir"]
environment = environment_report()

import genesis as gs

backend = gs.cpu if backend_mode == "cpu" else select_backend(prefer_rocm=True)
gs.init(backend=backend, seed=0, precision="32", logging_level="warning")

if getattr(gs, "amdgpu", None) is not None and gs.backend == gs.amdgpu:
    actual_backend = "amdgpu"
elif gs.backend == gs.cpu:
    actual_backend = "cpu"
else:
    actual_backend = str(gs.backend)

print(f"{'requested backend':>24}: {backend_mode}")
print(f"{'actual backend':>24}: {actual_backend}")
print(f"{'render enabled':>24}: {render_enabled}")
print(f"{'output directory':>24}: {output_dir.resolve()}")


## 控制前先预测

构建场景前，先写下你的答案：

1. 固定关系会增加可控 DOF 吗？
2. 为什么一个 7 关节机械臂会产生 9 维整机命令？
3. `set_dofs_position` 和 `control_dofs_position` 会产生相同的运动历史吗？
4. G1→G2 只提高 joint4 KP 时，哪些量可能变化？
5. G2→G3 只提高 joint4 KV 时，哪些量可能变化？
6. 哪些证据需要 Genesis camera，哪些结论必须依靠时间序列数组？


In [ ]:
DT = 0.01
SUBSTEPS = 2
SIM_DURATION = 1.2
N_STEPS = round(SIM_DURATION / DT)
assert N_STEPS > 0
assert np.isclose(N_STEPS * DT, SIM_DURATION, rtol=0.0, atol=1e-12)

scene = gs.Scene(
    sim_options=gs.options.SimOptions(dt=DT, substeps=SUBSTEPS),
    rigid_options=gs.options.RigidOptions(enable_collision=True),
    show_viewer=False,
)
scene.add_entity(gs.morphs.Plane())
franka = scene.add_entity(gs.morphs.MJCF(file=FRANKA_MJCF))

camera = None
if render_enabled:
    camera = scene.add_camera(
        res=(720, 540),
        pos=(1.8, -1.8, 1.4),
        lookat=(0.0, 0.0, 0.55),
        fov=42,
        GUI=False,
    )

scene.build()
print(
    f"built Franka: links={len(franka.links)}, joints={len(franka.joints)}, "
    f"dofs={franka.n_dofs}, qpos={franka.n_qs}"
)
print(f"outer dt={DT:.3f} s; substeps={SUBSTEPS}; internal dt={DT / SUBSTEPS:.4f} s")
if render_enabled:
    print("render path: Genesis camera declared before build")
else:
    print("render path: SKIP — ROBO_GENESIS_RENDER=0")


## Part A · 先检查模型，再执行一次基准运动

### 发送命令前先阅读结构

Link 是刚体，Joint 约束相对运动，DOF 是独立的标量运动轴，qpos 保存广义位形；它们的索引不能互换。固定关系不贡献 DOF，7 个臂关节是 revolute joint，两个 finger joint 是 prismatic joint。

下一个 cell 使用 `get_joint(name).dofs_idx_local` 解析每个受控维度，同时读取 `n_dofs`、`n_qs`、joint 类型、qpos index、单位和位置限制。这个固定基座模型恰好有 9 个 DOF 和 9 个 qpos 坐标；free joint 和 spherical joint 说明了这种相等关系并不普遍成立。


In [ ]:
JOINT_NAMES = [f"joint{i}" for i in range(1, 8)] + [
    "finger_joint1",
    "finger_joint2",
]
EXPECTED_TYPES = ["revolute"] * 7 + ["prismatic"] * 2
POSITION_UNITS = ["rad"] * 7 + ["m"] * 2
VELOCITY_UNITS = ["rad/s"] * 7 + ["m/s"] * 2
EFFORT_UNITS = ["N·m"] * 7 + ["N"] * 2

joint_records = []
dof_indices = []
q_indices = []
for name in JOINT_NAMES:
    joint = franka.get_joint(name)
    if joint.n_dofs != 1 or joint.n_qs != 1:
        raise AssertionError(
            f"{name}: expected one DOF and one qpos, got "
            f"n_dofs={joint.n_dofs}, n_qs={joint.n_qs}"
        )
    local_dofs = list(joint.dofs_idx_local)
    local_qs = list(joint.qs_idx_local)
    dof_indices.extend(local_dofs)
    q_indices.extend(local_qs)
    joint_records.append(
        {
            "name": name,
            "type": joint.type.name.lower(),
            "n_dofs": joint.n_dofs,
            "n_qs": joint.n_qs,
            "dof": local_dofs[0],
            "qpos": local_qs[0],
        }
    )

all_dofs = np.asarray(dof_indices, dtype=int)
all_qs = np.asarray(q_indices, dtype=int)
arm_dofs = all_dofs[:7]
finger_dofs = all_dofs[7:]
limit_lower, limit_upper = (
    to_numpy(value).reshape(-1).astype(float)
    for value in franka.get_dofs_limit(dofs_idx_local=all_dofs)
)
runtime_qpos = to_numpy(franka.get_qpos(qs_idx_local=all_qs)).reshape(-1)

structure_checks = {
    "eleven_links": len(franka.links) == 11,
    "nine_joints": len(franka.joints) == 9,
    "nine_dofs": franka.n_dofs == 9 and all_dofs.shape == (9,),
    "nine_qpos": franka.n_qs == 9 and runtime_qpos.shape == (9,),
    "unique_local_dofs": np.array_equal(np.sort(all_dofs), np.arange(9)),
    "unique_local_qpos": np.array_equal(np.sort(all_qs), np.arange(9)),
    "seven_arm_two_finger": arm_dofs.shape == (7,) and finger_dofs.shape == (2,),
    "expected_joint_types": [row["type"] for row in joint_records] == EXPECTED_TYPES,
    "finite_position_limits": np.isfinite(limit_lower).all() and np.isfinite(limit_upper).all(),
}
failed_structure = [name for name, passed in structure_checks.items() if not passed]
if failed_structure:
    raise AssertionError("Franka structure checks failed: " + ", ".join(failed_structure))

lines = [
    "| Joint | type | n_dofs | n_qs | local DOF | local qpos | position range | position / velocity / effort |",
    "|---|---|---:|---:|---:|---:|---:|---|",
]
for index, row in enumerate(joint_records):
    lines.append(
        f"| {row['name']} | {row['type']} | {row['n_dofs']} | {row['n_qs']} | "
        f"{row['dof']} | {row['qpos']} | [{limit_lower[index]:.4f}, "
        f"{limit_upper[index]:.4f}] | {POSITION_UNITS[index]} / "
        f"{VELOCITY_UNITS[index]} / {EFFORT_UNITS[index]} |"
    )
display(Markdown("\n".join(lines)))
print(structure_checks)


### 动态控制前先重置状态

`set_dofs_position(..., zero_velocity=True)` 会直接建立可重复的初始位置和零速度，但它不能证明机器人通过执行器完成了运动。`control_dofs_position(...)` 设置位置目标；只有 `scene.step()` 推进动力学后，实测状态才会变化。

Genesis 还提供 `control_dofs_velocity(...)` 来设置速度目标，提供 `control_dofs_force(...)` 来发送广义力或力矩命令。本实验解释这些接口边界，但只实际执行位置控制。

对于目标速度为零的情况，可以使用下面这个有边界的心智模型：

`tau_control ≈ KP × (q_target - q) - KV × qdot`。

它只能预测可能的变化趋势，不是完整的 Franka 模型。有效惯量随位姿变化，关节之间存在耦合，重力和离散时间也会影响响应，`set_dofs_force_range(...)` 还会截断控制贡献。基准实验先在 t=0 记录一个 q 样本，然后在每个外层 step 后记录 q、qdot 和 `get_dofs_control_force()`。


In [ ]:
q_start = np.asarray(FRANKA_QPOS, dtype=float)
baseline_kp = np.asarray(FRANKA_KP, dtype=float)
baseline_kv = np.asarray(FRANKA_KV, dtype=float)
force_lower = np.asarray(FRANKA_FORCE_MIN, dtype=float)
force_upper = np.asarray(FRANKA_FORCE_MAX, dtype=float)
# Runtime limits are float32; this tolerance handles boundary representation only.
LIMIT_TOLERANCE = 1e-6
for name, values in {
    "q_start": q_start,
    "baseline_kp": baseline_kp,
    "baseline_kv": baseline_kv,
    "force_lower": force_lower,
    "force_upper": force_upper,
}.items():
    assert values.shape == (9,), (name, values.shape)
    assert np.isfinite(values).all(), name
assert np.all(limit_lower - LIMIT_TOLERANCE <= q_start)
assert np.all(q_start <= limit_upper + LIMIT_TOLERANCE)

POSE_LINK_NAMES = [f"link{i}" for i in range(8)] + [
    "hand",
    "left_finger",
    "right_finger",
]


def read_dofs(getter):
    values = to_numpy(getter(dofs_idx_local=all_dofs)).reshape(-1).astype(float)
    if values.shape != (9,) or not np.isfinite(values).all():
        raise AssertionError(f"invalid DOF state from {getter.__name__}: {values}")
    return values.copy()


def read_link_positions():
    positions = {
        name: to_numpy(franka.get_link(name).get_pos()).reshape(-1).astype(float).copy()
        for name in POSE_LINK_NAMES
    }
    for name, position in positions.items():
        if position.shape != (3,) or not np.isfinite(position).all():
            raise AssertionError(f"{name}: invalid measured link position {position}")
    return positions


def render_rgb(label):
    if camera is None:
        return None
    rgb = to_numpy(camera.render(rgb=True)[0])
    if rgb.ndim != 3 or rgb.shape[0] == 0 or rgb.shape[1] == 0 or rgb.shape[2] not in (3, 4):
        raise AssertionError(f"{label}: expected non-empty HxWx3/4 RGB, got {rgb.shape}")
    if not np.isfinite(rgb).all():
        raise AssertionError(f"{label}: RGB contains non-finite values")
    return rgb.copy()


franka.set_dofs_kp(baseline_kp, dofs_idx_local=all_dofs)
franka.set_dofs_kv(baseline_kv, dofs_idx_local=all_dofs)
franka.set_dofs_force_range(force_lower, force_upper, dofs_idx_local=all_dofs)
measured_force_lower, measured_force_upper = (
    to_numpy(value).reshape(-1).astype(float)
    for value in franka.get_dofs_force_range(dofs_idx_local=all_dofs)
)
assert np.allclose(measured_force_lower, force_lower)
assert np.allclose(measured_force_upper, force_upper)

franka.set_dofs_position(q_start, dofs_idx_local=all_dofs, zero_velocity=True)
initial_q = read_dofs(franka.get_dofs_position)
initial_qdot = read_dofs(franka.get_dofs_velocity)
assert np.allclose(initial_q, q_start, rtol=0.0, atol=1e-6)
assert np.allclose(initial_qdot, 0.0, rtol=0.0, atol=1e-7)
initial_link_positions = read_link_positions()
initial_rgb = render_rgb("baseline initial")

joint4_vector_index = JOINT_NAMES.index("joint4")
q_target = q_start.copy()
q_target[joint4_vector_index] += 0.25
assert np.all(limit_lower - LIMIT_TOLERANCE <= q_target)
assert np.all(q_target <= limit_upper + LIMIT_TOLERANCE)

baseline_q = [initial_q]
baseline_qdot = []
baseline_control = []
for _ in range(N_STEPS):
    franka.control_dofs_position(q_target, dofs_idx_local=all_dofs)
    scene.step()
    baseline_q.append(read_dofs(franka.get_dofs_position))
    baseline_qdot.append(read_dofs(franka.get_dofs_velocity))
    baseline_control.append(read_dofs(franka.get_dofs_control_force))

baseline_q = np.asarray(baseline_q)
baseline_qdot = np.asarray(baseline_qdot)
baseline_control = np.asarray(baseline_control)
q_time = np.arange(N_STEPS + 1, dtype=float) * DT
step_time = np.arange(1, N_STEPS + 1, dtype=float) * DT

assert baseline_q.shape == (N_STEPS + 1, 9)
assert baseline_qdot.shape == (N_STEPS, 9)
assert baseline_control.shape == (N_STEPS, 9)
assert np.isfinite(baseline_q).all()
assert np.isfinite(baseline_qdot).all()
assert np.isfinite(baseline_control).all()

final_link_positions = read_link_positions()
final_rgb = render_rgb("baseline final")
baseline_initial_error = abs(q_target[joint4_vector_index] - baseline_q[0, joint4_vector_index])
baseline_final_error = abs(q_target[joint4_vector_index] - baseline_q[-1, joint4_vector_index])
print(f"joint4 target: {q_target[joint4_vector_index]:.4f} rad")
print(f"initial error: {baseline_initial_error:.6f} rad")
print(f"final error:   {baseline_final_error:.6f} rad")
print(f"final speed:   {abs(baseline_qdot[-1, joint4_vector_index]):.6f} rad/s")


In [ ]:
ARM_LINK_NAMES = [f"link{i}" for i in range(8)] + ["hand"]
FINGER_LINK_NAMES = ["left_finger", "right_finger"]


def draw_robot_schematic(axis, positions, title, color, limits):
    arm = np.vstack([positions[name] for name in ARM_LINK_NAMES])
    axis.plot(arm[:, 0], arm[:, 1], arm[:, 2], "-o", color=color, label="arm links")
    hand = positions["hand"]
    for index, name in enumerate(FINGER_LINK_NAMES):
        finger = positions[name]
        axis.plot(
            [hand[0], finger[0]],
            [hand[1], finger[1]],
            [hand[2], finger[2]],
            "-o",
            color="#555555",
            label="finger links" if index == 0 else None,
        )
    axis.set(
        xlim=limits[0],
        ylim=limits[1],
        zlim=limits[2],
        xlabel="x [m]",
        ylabel="y [m]",
        zlabel="z [m]",
        title=title,
    )
    axis.set_box_aspect((1.0, 1.0, 1.2))
    axis.view_init(elev=22, azim=-58)
    axis.legend(fontsize=8)


all_link_positions = np.vstack([
    *initial_link_positions.values(),
    *final_link_positions.values(),
])
center = 0.5 * (all_link_positions.min(axis=0) + all_link_positions.max(axis=0))
half_extent = max(0.35, 0.6 * float(np.max(np.ptp(all_link_positions, axis=0))))
schematic_limits = tuple(
    (float(value - half_extent), float(value + half_extent))
    for value in center
)

pose_titles = (
    f"Initial pose: joint4 = {baseline_q[0, joint4_vector_index]:.3f} rad",
    f"Final pose: joint4 = {baseline_q[-1, joint4_vector_index]:.3f} rad",
)
if render_enabled:
    pose_figure, pose_axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for axis, frame, title in zip(pose_axes, (initial_rgb, final_rgb), pose_titles):
        axis.imshow(frame)
        axis.set_title(title)
        axis.axis("off")
    visualization_mode = "Genesis camera frames"
else:
    pose_figure, pose_axes = plt.subplots(
        1,
        2,
        figsize=(11, 4.8),
        subplot_kw={"projection": "3d"},
    )
    draw_robot_schematic(
        pose_axes[0],
        initial_link_positions,
        pose_titles[0],
        "#247BA0",
        schematic_limits,
    )
    draw_robot_schematic(
        pose_axes[1],
        final_link_positions,
        pose_titles[1],
        "#D95F43",
        schematic_limits,
    )
    visualization_mode = "measured-link schematics — rendering disabled"

pose_suptitle = "Franka posture before and after dynamic position control"
if not render_enabled:
    pose_suptitle += " — measured-link schematics, not camera frames"
pose_figure.suptitle(pose_suptitle)
pose_figure.tight_layout()
pose_path = output_dir / "franka_initial_final.png"
pose_figure.savefig(pose_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(pose_figure)

baseline_figure, baseline_axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
baseline_axes[0].plot(q_time, baseline_q[:, joint4_vector_index], label="measured joint4")
baseline_axes[0].axhline(
    q_target[joint4_vector_index],
    color="tab:orange",
    linestyle="--",
    label="target",
)
baseline_axes[0].set(ylabel="position [rad]", title="Baseline position tracking")
baseline_axes[1].plot(
    step_time,
    baseline_qdot[:, joint4_vector_index],
    color="tab:green",
    label="measured joint4 velocity",
)
baseline_axes[1].axhline(0.0, color="#444444", linewidth=0.8)
baseline_axes[1].set(ylabel="velocity [rad/s]", title="Measured joint velocity")
baseline_axes[2].plot(
    step_time,
    baseline_control[:, joint4_vector_index],
    color="tab:red",
    label="measured control contribution",
)
baseline_axes[2].axhline(
    force_upper[joint4_vector_index],
    color="#444444",
    linestyle="--",
    label="force range",
)
baseline_axes[2].axhline(force_lower[joint4_vector_index], color="#444444", linestyle="--")
baseline_axes[2].set(
    xlabel="simulated time [s]",
    ylabel="control torque [N·m]",
    title="Controller contribution and configured range",
)
for axis in baseline_axes:
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
baseline_figure.tight_layout()
baseline_path = output_dir / "joint4_baseline_response.png"
baseline_figure.savefig(baseline_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(baseline_figure)

print("posture evidence:", visualization_mode)
print("saved:", pose_path.resolve())
print("saved:", baseline_path.resolve())


## Part B · 隔离 KP 与 KV

三组 case 使用相同的模型、seed、q_start、零初始速度、joint4 +0.25 rad target、dt、substeps、duration、非 joint4 增益和 force range。

| Case | joint4 KP | joint4 KV | 唯一的对照变化 |
|---|---:|---:|---|
| G1 · Reference | 3500 | 100 | 低刚度参考组 |
| G2 · Higher KP only | 7000 | 100 | G1→G2 只提高 KP |
| G3 · More damping | 7000 | 300 | G2→G3 只提高 KV |

先做预测：G2 到达 90% 的时间可能不晚于 G1，但也可能出现更大过冲；G3 相对 G2 可能减小过冲和峰值速度，同时也可能让上升变慢。这些只是对当前位姿和有界执行器的假设，不是无条件成立的结果。

### 先阅读指标定义，再查看结果

- **rise：** 第一次到达本次正向阶跃 90% 的样本；否则报告 `not observed`；
- **overshoot：** 超过正向 target 的最大采样角度，下限截为零；
- **settling：** 第一次进入 ±0.01 rad，且全部后续样本都保持在该范围内的时刻；否则报告 `not observed`；
- **final error：** 最后一个样本的绝对 target error，必须与最终速度和 settling 一起解释；
- **peak speed / peak control：** 实测 qdot 和控制贡献的最大绝对值；
- **saturation：** 控制绝对值至少达到读回 joint4 force limit 99% 的样本。

Settling 只是有限 1.2 秒窗口内的结论，不能证明无限时域稳定。`get_dofs_control_force()` 是本实验用于 limit 检查的控制器贡献；`get_dofs_force()` 表示 DOF 实际承受的内部力，不能在这里替换前者。


In [ ]:
SETTLING_TOLERANCE = 0.01
SATURATION_RATIO = 0.99
GAIN_CASES = [
    {"id": "G1", "label": "G1 · Reference", "kp": 3500.0, "kv": 100.0},
    {"id": "G2", "label": "G2 · Higher KP only", "kp": 7000.0, "kv": 100.0},
    {"id": "G3", "label": "G3 · More damping", "kp": 7000.0, "kv": 300.0},
]


def first_sustained_index(mask):
    values = np.asarray(mask, dtype=bool)
    if values.ndim != 1 or values.size == 0:
        raise ValueError("mask must be a non-empty 1-D array")
    suffix_all_true = np.logical_and.accumulate(values[::-1])[::-1]
    indices = np.flatnonzero(suffix_all_true)
    return int(indices[0]) if indices.size else None


def step_response_metrics(q_values, velocity_values, control_values, target_q, start_q, force_limit):
    q_values = np.asarray(q_values, dtype=float)
    velocity_values = np.asarray(velocity_values, dtype=float)
    control_values = np.asarray(control_values, dtype=float)
    if q_values.shape != (N_STEPS + 1,):
        raise ValueError(f"unexpected q shape: {q_values.shape}")
    if velocity_values.shape != (N_STEPS,) or control_values.shape != (N_STEPS,):
        raise ValueError(
            f"unexpected velocity/control shapes: {velocity_values.shape}, {control_values.shape}"
        )
    if not all(np.isfinite(values).all() for values in (q_values, velocity_values, control_values)):
        raise ValueError("step response contains non-finite values")
    if target_q <= start_q:
        raise ValueError("this lesson's one-sided metrics require a positive step")

    rise_threshold = start_q + 0.9 * (target_q - start_q)
    rise_indices = np.flatnonzero(q_values >= rise_threshold)
    rise_time = q_time[rise_indices[0]] if rise_indices.size else np.nan
    overshoot = max(0.0, float(np.max(q_values - target_q)))
    settling_index = first_sustained_index(
        np.abs(q_values - target_q) <= SETTLING_TOLERANCE
    )
    settling_time = q_time[settling_index] if settling_index is not None else np.nan
    saturation_mask = np.abs(control_values) >= SATURATION_RATIO * force_limit
    return {
        "rise_time": float(rise_time),
        "overshoot": overshoot,
        "settling_time": float(settling_time),
        "final_error": abs(float(target_q - q_values[-1])),
        "final_speed": abs(float(velocity_values[-1])),
        "peak_speed": float(np.max(np.abs(velocity_values))),
        "peak_control": float(np.max(np.abs(control_values))),
        "saturation_samples": int(np.count_nonzero(saturation_mask)),
        "saturation_duration": float(np.count_nonzero(saturation_mask) * DT),
    }


def run_gain_case(spec):
    franka.set_dofs_position(q_start, dofs_idx_local=all_dofs, zero_velocity=True)
    reset_q = read_dofs(franka.get_dofs_position)
    reset_qdot = read_dofs(franka.get_dofs_velocity)
    if not np.allclose(reset_q, q_start, rtol=0.0, atol=1e-6):
        raise AssertionError(f"{spec['id']}: position reset failed")
    if not np.allclose(reset_qdot, 0.0, rtol=0.0, atol=1e-7):
        raise AssertionError(f"{spec['id']}: velocity reset failed")

    kp_vector = baseline_kp.copy()
    kv_vector = baseline_kv.copy()
    kp_vector[joint4_vector_index] = spec["kp"]
    kv_vector[joint4_vector_index] = spec["kv"]
    franka.set_dofs_kp(kp_vector, dofs_idx_local=all_dofs)
    franka.set_dofs_kv(kv_vector, dofs_idx_local=all_dofs)
    franka.set_dofs_force_range(force_lower, force_upper, dofs_idx_local=all_dofs)

    case_target = q_start.copy()
    case_target[joint4_vector_index] += 0.25
    inside_limits = (limit_lower - LIMIT_TOLERANCE <= case_target) & (
        case_target <= limit_upper + LIMIT_TOLERANCE
    )
    if not np.all(inside_limits):
        raise AssertionError(f"{spec['id']}: target exceeds a position limit")

    q_values = [reset_q[joint4_vector_index]]
    velocity_values = []
    control_values = []
    for _ in range(N_STEPS):
        franka.control_dofs_position(case_target, dofs_idx_local=all_dofs)
        scene.step()
        q_values.append(read_dofs(franka.get_dofs_position)[joint4_vector_index])
        velocity_values.append(read_dofs(franka.get_dofs_velocity)[joint4_vector_index])
        control_values.append(read_dofs(franka.get_dofs_control_force)[joint4_vector_index])

    q_values = np.asarray(q_values, dtype=float)
    velocity_values = np.asarray(velocity_values, dtype=float)
    control_values = np.asarray(control_values, dtype=float)
    metrics = step_response_metrics(
        q_values,
        velocity_values,
        control_values,
        case_target[joint4_vector_index],
        q_start[joint4_vector_index],
        max(
            abs(measured_force_lower[joint4_vector_index]),
            abs(measured_force_upper[joint4_vector_index]),
        ),
    )
    return {
        **spec,
        "q_start": q_start.copy(),
        "target": case_target.copy(),
        "kp_vector": kp_vector,
        "kv_vector": kv_vector,
        "force_lower": force_lower.copy(),
        "force_upper": force_upper.copy(),
        "dt": DT,
        "substeps": SUBSTEPS,
        "n_steps": N_STEPS,
        "q": q_values,
        "velocity": velocity_values,
        "control_force": control_values,
        **metrics,
    }


gain_results = {spec["id"]: run_gain_case(spec) for spec in GAIN_CASES}
franka.set_dofs_kp(baseline_kp, dofs_idx_local=all_dofs)
franka.set_dofs_kv(baseline_kv, dofs_idx_local=all_dofs)

g1, g2, g3 = (gain_results[case_id] for case_id in ("G1", "G2", "G3"))
controlled_variable_checks = {
    "G1_to_G2_only_joint4_kp": (
        np.flatnonzero(g1["kp_vector"] != g2["kp_vector"]).tolist() == [joint4_vector_index]
        and np.array_equal(g1["kv_vector"], g2["kv_vector"])
    ),
    "G2_to_G3_only_joint4_kv": (
        np.array_equal(g2["kp_vector"], g3["kp_vector"])
        and np.flatnonzero(g2["kv_vector"] != g3["kv_vector"]).tolist() == [joint4_vector_index]
    ),
    "fixed_initial_target_and_limits": all(
        np.array_equal(result["q_start"], g1["q_start"])
        and np.array_equal(result["target"], g1["target"])
        and np.array_equal(result["force_lower"], g1["force_lower"])
        and np.array_equal(result["force_upper"], g1["force_upper"])
        and result["dt"] == g1["dt"]
        and result["substeps"] == g1["substeps"]
        and result["n_steps"] == g1["n_steps"]
        for result in gain_results.values()
    ),
}
assert all(controlled_variable_checks.values()), controlled_variable_checks


def format_optional(value, digits=3):
    return "not observed" if not np.isfinite(value) else f"{value:.{digits}f}"


lines = [
    "| Case | KP | KV | rise [s] | overshoot [rad] | settling [s] | final error [rad] | peak speed [rad/s] | peak control [N·m] | saturation [ms] |",
    "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
]
for spec in GAIN_CASES:
    result = gain_results[spec["id"]]
    lines.append(
        f"| {result['label']} | {result['kp']:.0f} | {result['kv']:.0f} | "
        f"{format_optional(result['rise_time'])} | {result['overshoot']:.4f} | "
        f"{format_optional(result['settling_time'])} | {result['final_error']:.5f} | "
        f"{result['peak_speed']:.3f} | {result['peak_control']:.2f} | "
        f"{1000 * result['saturation_duration']:.1f} |"
    )
display(Markdown("\n".join(lines)))
print(controlled_variable_checks)


In [ ]:
GAIN_COLORS = {"G1": "#247BA0", "G2": "#D95F43", "G3": "#2A9D5B"}
gain_figure, gain_axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
for case_id, result in gain_results.items():
    label = f"{case_id}: KP={result['kp']:.0f}, KV={result['kv']:.0f}"
    gain_axes[0].plot(q_time, result["q"], color=GAIN_COLORS[case_id], label=label)
    gain_axes[1].plot(step_time, result["velocity"], color=GAIN_COLORS[case_id], label=case_id)
    gain_axes[2].plot(step_time, result["control_force"], color=GAIN_COLORS[case_id], label=case_id)

gain_axes[0].axhline(
    g1["target"][joint4_vector_index],
    color="black",
    linestyle="--",
    linewidth=1.2,
    label="target",
)
gain_axes[0].set(ylabel="joint4 position [rad]", title="Position step response")
gain_axes[1].axhline(0.0, color="black", linewidth=0.8)
gain_axes[1].set(ylabel="joint4 velocity [rad/s]", title="Velocity and damping")
gain_axes[2].axhline(
    measured_force_upper[joint4_vector_index],
    color="black",
    linestyle="--",
    linewidth=1.0,
    label="force range",
)
gain_axes[2].axhline(
    measured_force_lower[joint4_vector_index],
    color="black",
    linestyle="--",
    linewidth=1.0,
)
gain_axes[2].set(
    xlabel="simulated time [s]",
    ylabel="control torque [N·m]",
    title="Controller contribution and saturation",
)
for axis in gain_axes:
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
gain_figure.tight_layout()
gain_path = output_dir / "kp_kv_step_response.png"
gain_figure.savefig(gain_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(gain_figure)
print("saved:", gain_path.resolve())


In [ ]:
def observed_time(value):
    return "not observed" if not np.isfinite(value) else f"{value:.3f} s"


kp_relation_supported = (
    np.isfinite(g1["rise_time"])
    and np.isfinite(g2["rise_time"])
    and g2["rise_time"] <= g1["rise_time"]
    and g2["overshoot"] > g1["overshoot"]
)
kv_relation_supported = (
    g3["overshoot"] < g2["overshoot"]
    and g3["peak_speed"] < g2["peak_speed"]
)
gain_relationship_checks = {
    "G2_rise_no_later_than_G1_and_more_overshoot": kp_relation_supported,
    "G3_less_overshoot_and_peak_speed_than_G2": kv_relation_supported,
    "at_least_one_case_reaches_force_range": any(
        result["saturation_samples"] > 0 for result in gain_results.values()
    ),
}

kp_verdict = (
    "The measured direction supports the local prediction for this run."
    if kp_relation_supported
    else "The run does not show both predicted directions; inspect saturation, timing, and the complete traces."
)
kv_verdict = (
    "The measured direction supports the damping prediction for this run."
    if kv_relation_supported
    else "The run does not show both predicted damping directions; keep the measured result and diagnose it."
)
settling_lines = "; ".join(
    f"{case_id}: final error={result['final_error']:.5f} rad, "
    f"final speed={result['final_speed']:.5f} rad/s, "
    f"settling={observed_time(result['settling_time'])}"
    for case_id, result in gain_results.items()
)
saturation_lines = "; ".join(
    f"{case_id}: {result['saturation_samples']} samples "
    f"({1000 * result['saturation_duration']:.1f} ms), "
    f"peak={result['peak_control']:.2f} N·m"
    for case_id, result in gain_results.items()
)
any_saturation = gain_relationship_checks["at_least_one_case_reaches_force_range"]
saturation_boundary = (
    "At least one trace reaches the configured range, so an unconstrained ideal-PD model "
    "does not explain the complete transient."
    if any_saturation
    else
    "No sampled control value reaches the declared threshold in this run; the force range "
    "still remains part of the experiment contract."
)

guided_interpretation = f"""
### Guided interpretation

**1. G1 → G2 isolates KP.** KV remains {g1['kv']:.0f}, while KP changes from
{g1['kp']:.0f} to {g2['kp']:.0f}. Rise is {observed_time(g1['rise_time'])} →
{observed_time(g2['rise_time'])}; overshoot is {g1['overshoot']:.5f} →
{g2['overshoot']:.5f} rad; peak speed is {g1['peak_speed']:.4f} →
{g2['peak_speed']:.4f} rad/s. {kp_verdict}

**2. G2 → G3 isolates KV.** KP remains {g2['kp']:.0f}, while KV changes from
{g2['kv']:.0f} to {g3['kv']:.0f}. Rise is {observed_time(g2['rise_time'])} →
{observed_time(g3['rise_time'])}; overshoot is {g2['overshoot']:.5f} →
{g3['overshoot']:.5f} rad; peak speed is {g2['peak_speed']:.4f} →
{g3['peak_speed']:.4f} rad/s. {kv_verdict}

**3. Final error has a finite-window boundary.** {settling_lines}. Read final
error together with final speed and finite-window settling; do not assign every
remaining error to gravity or use one endpoint to describe the transient.

**4. Actuator limits bound the interpretation.** The joint4 range is
[{measured_force_lower[joint4_vector_index]:.1f},
{measured_force_upper[joint4_vector_index]:.1f}] N·m. {saturation_lines}.
{saturation_boundary}

These statements were generated from the current arrays. They apply to the
reported Genesis version, backend, pose, target, timestep, force range, and
observation window; they are not a universal Franka tuning recipe.
"""
display(Markdown(guided_interpretation))
print(gain_relationship_checks)


## 进入 L05 前的检查点

根据你生成的结构表、camera/state 证据、曲线、指标和 Guided interpretation 回答：

1. 固定关系为什么不贡献 DOF？
2. 为什么当前 Franka 有 7 个 arm DOF，却有 9 个受控维度？
3. `set_dofs_position` 和 `control_dofs_position` 有什么区别？
4. 提高 KP 后，为什么还必须重新检查 KV、速度、过冲和 force range？
5. 为什么只看 final error 既无法描述瞬态，也不能证明误差只来自重力？
6. 哪些证据来自 Genesis 渲染，哪些结论来自实测数组？

运行最终检查前，先用自己的话解释每个答案。


In [ ]:
baseline_checks = {
    "baseline_position_shape": baseline_q.shape == (N_STEPS + 1, 9),
    "baseline_velocity_shape": baseline_qdot.shape == (N_STEPS, 9),
    "baseline_control_shape": baseline_control.shape == (N_STEPS, 9),
    "baseline_values_finite": (
        np.isfinite(baseline_q).all()
        and np.isfinite(baseline_qdot).all()
        and np.isfinite(baseline_control).all()
    ),
    "baseline_error_decreased": baseline_final_error < baseline_initial_error,
}
gain_data_checks = {
    f"{case_id}_arrays_valid": (
        result["q"].shape == (N_STEPS + 1,)
        and result["velocity"].shape == (N_STEPS,)
        and result["control_force"].shape == (N_STEPS,)
        and np.isfinite(result["q"]).all()
        and np.isfinite(result["velocity"]).all()
        and np.isfinite(result["control_force"]).all()
    )
    for case_id, result in gain_results.items()
}
visual_checks = {
    "measured_link_states_valid": all(
        position.shape == (3,) and np.isfinite(position).all()
        for positions in (initial_link_positions, final_link_positions)
        for position in positions.values()
    ),
    "render_branch_explicit": (
        initial_rgb is not None and final_rgb is not None
        if render_enabled
        else initial_rgb is None and final_rgb is None
    ),
}
runtime_checks = {
    "genesis_1_3_3": environment["genesis_world"] == "1.3.3",
    "actual_backend_supported": actual_backend in {"cpu", "amdgpu"},
    "forced_cpu_honored": backend_mode != "cpu" or actual_backend == "cpu",
    "timing_exact": N_STEPS == 120 and np.isclose(q_time[-1], SIM_DURATION),
    "target_inside_limits": (
        np.all(limit_lower - LIMIT_TOLERANCE <= q_target)
        and np.all(q_target <= limit_upper + LIMIT_TOLERANCE)
    ),
    "force_range_readback": (
        np.allclose(measured_force_lower, force_lower)
        and np.allclose(measured_force_upper, force_upper)
    ),
}
all_checks = {
    **structure_checks,
    **runtime_checks,
    **baseline_checks,
    **controlled_variable_checks,
    **gain_data_checks,
    **gain_relationship_checks,
    **visual_checks,
}
for name, passed in all_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")

failed_checks = [name for name, passed in all_checks.items() if not passed]
print("requested backend:", backend_mode)
print("actual backend:", actual_backend)
print(
    "rendering:",
    "PASSED — Genesis initial/final RGB captured"
    if render_enabled
    else "SKIP — disabled before build; measured-link schematics used",
)
print("evidence directory:", output_dir.resolve())
if failed_checks:
    raise AssertionError("L04 checks failed: " + ", ".join(failed_checks))

evidence_summary = f"""
### Evidence summary generated by this run

- Runtime structure: {len(franka.links)} Links, {len(franka.joints)} Joints,
  {franka.n_dofs} DOFs, and {franka.n_qs} qpos coordinates; the named mapping
  resolves seven arm DOFs and two finger DOFs.
- Baseline: joint4 error changed from {baseline_initial_error:.6f} to
  {baseline_final_error:.6f} rad through dynamic position control.
- Gain comparison: the one-factor checks passed; measured relationships and
  force-range evidence are reported in the Guided interpretation above.
- Visual path: {visualization_mode}. Camera frames are scene evidence; q,
  qdot, control-force traces, and metrics provide the transient evidence.

L05 will compute joint targets from an end-effector pose. Those targets still
need this target → controller → dynamics → measured-state loop to become motion.
"""
display(Markdown(evidence_summary))
print("L04 CHECK: PASSED")
